In [ ]:
# Install Klygo from the v2-release branch
!pip install --upgrade --no-cache-dir git+https://github.com/IchigoMazone/klygo.git@v2-release

# Mining Curation

[!["Open In Colab"](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/mining/Document_Mining_Curation.ipynb)

Perform deterministic clustering, diverse sample selection, and class imbalance mitigation.

## Jump to an API

- [mining.cluster](#mining.cluster)
- [mining.select](#mining.select)
- [mining.balance](#mining.balance)
- [mining.ClusterResult](#mining.ClusterResult)

### mining.cluster

Group images using histograms, HOG, quality, boxes, and class presence.

In [ ]:
from klygo import datasets, mining

# Load a materialized YOLO dataset
data = datasets.load("artifacts/tutorials/mining/flat_coco8", format="yolo", verbose=False)

# Cluster dataset into visual groups
clusters = mining.cluster(data, clusters=min(3, len(data)), seed=42)
clusters.show()
print("Cluster representatives:", clusters.representatives)

### mining.select

Select whole images by class, quality, boxes, random sampling, quota, rarity, diversity, or score.

In [ ]:
# Select using random sampling
sel_rand = mining.select(data, method="random", limit=4, seed=42)
print(f"Random selection: {len(sel_rand)} images")

# Select using visual cluster diversity
sel_div = mining.select(data, method="diversity", cluster_result=clusters, limit=3)
print(f"Diversity selection: {len(sel_div)} images")

# Select using multi-criteria weighted scoring
scores = mining.score(data)
sel_weight = mining.select(data, method="weighted", scores=scores, limit=4)
print(f"Weighted selection: {len(sel_weight)} images")

### mining.balance

Reduce imbalance without creating duplicate frame identities.

In [ ]:
# Balance dataset targeting median class count
balanced = mining.balance(data, target="median", count_by="images", seed=42)
print(f"Original {len(data)} -> Balanced {len(balanced)} images")

### mining.ClusterResult

Cluster assignments, groups, and representatives returned by `cluster`.

In [ ]:
clusters.save("artifacts/clusters.json")
print("Saved ClusterResult to JSON.")

## Put it together

Cluster images by visual features, select diverse representatives, and balance class distributions.

In [ ]:
clusters = mining.cluster(data, clusters=min(3, len(data)), seed=42)
curated = mining.select(data, method="diversity", cluster_result=clusters, limit=3)
balanced = mining.balance(curated, target="median", seed=42)
balanced.export("artifacts/curated_balanced", format="yolo", overwrite=True)
print("Curated and balanced dataset exported.")

## Remember

1. `cluster` uses deterministic K-Means over normalized visual and annotation features.
2. `select` supports 9 curation strategies: random, weighted, quality, rare_first, diversity, boxes, quota, coverage, and class.
3. `balance` reduces class imbalance without synthesizing or duplicating images.